# 25 · Autoencodeurs et VAE — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch25_autoencodeurs/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch25_autoencodeurs/03_notebook.ipynb`, créée par `python tools/start_chapter.py 25`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 25.Q1 à 25.Q12, 25.R1 à 25.R3, 25.1 à 25.4, 25.6, 25.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 25.14 à 25.23 | des autoencodeurs denses : une seule image, MNIST, l'espace latent, l'interpolation, la taille du goulot, l'autoencodeur linéaire face à l'ACP (*prochaine session de génération*) | | | |
| B | 25.24 à 25.27 | des autoencodeurs convolutifs et débruiteurs : 147 latents, un 🐛 de débruiteur, sur-échantillonnage ou convolutions à pas (*prochaine session de génération*) | | | |
| C | 25.28 à 25.39 | le VAE : ta librairie `vae.py`, un VAE PyTorch, générer, β, les anomalies, un VAE en NumPy, le défi de débruitage (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="25")  # your own library

## Objectifs et rappel express

- Construire et entraîner en PyTorch des autoencodeurs denses, convolutifs et débruiteurs, et explorer leur espace latent (carte 2D, bruit, interpolation).
- Relier l'autoencodeur linéaire à l'ACP ; mesurer un débruitage (PSNR).
- Écrire en NumPy le cœur du VAE (`mylearn.vae`), entraîner un VAE, régler β, générer, détecter des anomalies.

**Rappel express.** $\mathbf{z} = f_\theta(\mathbf{x})$, $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$, la cible est l'entrée. Convolution transposée : $H_{out} = (H_{in} - 1)s - 2p + k + p_{out}$. PSNR $= 10 \log_{10}(1/\text{MSE})$ pour des pixels dans $[0, 1]$. VAE : $\mathbf{z} = \boldsymbol{\mu} + e^{\frac{1}{2}\log\boldsymbol{\sigma}^2} \odot \boldsymbol{\varepsilon}$, loss = reconstruction + $\beta$ KL, $\mathrm{KL} = \tfrac{1}{2}\sum_j (\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2)$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 25.1 à 25.4 et 25.6, ∂ 25.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`) ; une forme ou un vecteur : entre guillemets (`"(3, 5, 1)"`, `"(1, -2)"`) ; un mot : entre guillemets (`"ABCD"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 25.5 et 25.8 et la démarche de ∂ 25.7, les estimations 🧮 25.9 et 25.10, la réflexion (🗣️ 25.11, ⚖️ 25.12, 📄 25.13) et l'entretien se corrigent avec `05_solutions.md`.

**25.Q1 — Avec ou sans perte : le test de l'inversibilité**

In [ ]:
# 25.Q1: the values you found on paper
answer_25_Q1a = ...  # a) the letters, e.g. "AF"
answer_25_Q1b = ...  # b) True or False
answer_25_Q1c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q1d = ...  # d) a number
answer_25_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q1a, answer_25_Q1b, answer_25_Q1c, answer_25_Q1d, answer_25_Q1e]):
    wb.check(f"25.Q1{letter}", answer)

**25.Q2 — Pourquoi JPEG et MP3 battent un autoencodeur généraliste**

In [ ]:
# 25.Q2: the values you found on paper
answer_25_Q2a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q2b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q2c = ...  # c) True or False
answer_25_Q2d = ...  # d) the letters, e.g. "AF"
answer_25_Q2e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_25_Q2a, answer_25_Q2b, answer_25_Q2c, answer_25_Q2d, answer_25_Q2e]):
    wb.check(f"25.Q2{letter}", answer)

**25.Q3 — Mélange de contenu ou mélange de paramètres ?**

In [ ]:
# 25.Q3: the values you found on paper
answer_25_Q3a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q3b = ...  # b) a word as a string, e.g. "ABCD"
answer_25_Q3c = ...  # c) True or False
answer_25_Q3d = ...  # d) the letter of your choice, e.g. "F"

for letter, answer in zip("abcd", [answer_25_Q3a, answer_25_Q3b, answer_25_Q3c, answer_25_Q3d]):
    wb.check(f"25.Q3{letter}", answer)

**25.Q4 — Goulot, code, variables latentes : vrai ou faux**

In [ ]:
# 25.Q4: the values you found on paper
answer_25_Q4a = ...  # a) True or False
answer_25_Q4b = ...  # b) True or False
answer_25_Q4c = ...  # c) True or False
answer_25_Q4d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q4a, answer_25_Q4b, answer_25_Q4c, answer_25_Q4d, answer_25_Q4e]):
    wb.check(f"25.Q4{letter}", answer)

**25.Q5 — Le réseau qui ne savait redessiner qu'une image**

In [ ]:
# 25.Q5: the values you found on paper
answer_25_Q5a = ...  # a) a whole number
answer_25_Q5b = ...  # b) a whole number
answer_25_Q5c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q5d = ...  # d) True or False
answer_25_Q5e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_25_Q5a, answer_25_Q5b, answer_25_Q5c, answer_25_Q5d, answer_25_Q5e]):
    wb.check(f"25.Q5{letter}", answer)

**25.Q6 — Profondeur ou nombre de latents : qui améliore quoi ?**

In [ ]:
# 25.Q6: the values you found on paper
answer_25_Q6a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q6b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q6c = ...  # c) the letters, e.g. "AF"
answer_25_Q6d = ...  # d) True or False
answer_25_Q6e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_25_Q6a, answer_25_Q6b, answer_25_Q6c, answer_25_Q6d, answer_25_Q6e]):
    wb.check(f"25.Q6{letter}", answer)

**25.Q7 — Du bruit dans l'espace latent d'un autoencodeur classique**

In [ ]:
# 25.Q7: the values you found on paper
answer_25_Q7a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q7b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q7c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q7d = ...  # d) True or False
answer_25_Q7e = ...  # e) the letter of your choice, e.g. "F"

for letter, answer in zip("abcde", [answer_25_Q7a, answer_25_Q7b, answer_25_Q7c, answer_25_Q7d, answer_25_Q7e]):
    wb.check(f"25.Q7{letter}", answer)

**25.Q8 — Lire une carte latente à deux dimensions**

In [ ]:
# 25.Q8: the values you found on paper
answer_25_Q8a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q8b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q8c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q8d = ...  # d) True or False
answer_25_Q8e = ...  # e) a vector as a string, e.g. "(1, -2)"

for letter, answer in zip("abcde", [answer_25_Q8a, answer_25_Q8b, answer_25_Q8c, answer_25_Q8d, answer_25_Q8e]):
    wb.check(f"25.Q8{letter}", answer)

**25.Q9 — Autoencodeur linéaire et ACP**

In [ ]:
# 25.Q9: the values you found on paper
answer_25_Q9a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q9b = ...  # b) True or False
answer_25_Q9c = ...  # c) True or False
answer_25_Q9d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q9e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_25_Q9a, answer_25_Q9b, answer_25_Q9c, answer_25_Q9d, answer_25_Q9e]):
    wb.check(f"25.Q9{letter}", answer)

**25.Q10 — Autoencodeurs convolutifs et débruiteurs**

In [ ]:
# 25.Q10: the values you found on paper
answer_25_Q10a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q10b = ...  # b) True or False
answer_25_Q10c = ...  # c) the letter of your choice, e.g. "F"
answer_25_Q10d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q10a, answer_25_Q10b, answer_25_Q10c, answer_25_Q10d, answer_25_Q10e]):
    wb.check(f"25.Q10{letter}", answer)

**25.Q11 — Une entrée jamais vue : un tigre chez les chiffres**

In [ ]:
# 25.Q11: the values you found on paper
answer_25_Q11a = ...  # a) the letter of your choice, e.g. "F"
answer_25_Q11b = ...  # b) True or False
answer_25_Q11c = ...  # c) True or False
answer_25_Q11d = ...  # d) the letter of your choice, e.g. "F"
answer_25_Q11e = ...  # e) the letters, e.g. "AF"

for letter, answer in zip("abcde", [answer_25_Q11a, answer_25_Q11b, answer_25_Q11c, answer_25_Q11d, answer_25_Q11e]):
    wb.check(f"25.Q11{letter}", answer)

**25.Q12 — Les trois exigences d'un VAE et l'astuce de reparamétrisation**

In [ ]:
# 25.Q12: the values you found on paper
answer_25_Q12a = ...  # a) the letters, e.g. "AF"
answer_25_Q12b = ...  # b) the letter of your choice, e.g. "F"
answer_25_Q12c = ...  # c) True or False
answer_25_Q12d = ...  # d) a number, 2 decimals
answer_25_Q12e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_25_Q12a, answer_25_Q12b, answer_25_Q12c, answer_25_Q12d, answer_25_Q12e]):
    wb.check(f"25.Q12{letter}", answer)

**25.R1 — Quelles augmentations garder pour des chiffres manuscrits ?**

In [ ]:
# 25.R1: the values you found on paper
answer_25_R1a = ...  # a) the letters, e.g. "AF"
answer_25_R1b = ...  # b) True or False
answer_25_R1c = ...  # c) True or False

for letter, answer in zip("abc", [answer_25_R1a, answer_25_R1b, answer_25_R1c]):
    wb.check(f"25.R1{letter}", answer)

**25.R2 — Résumer une séquence entière dans l'état final d'un LSTM**

In [ ]:
# 25.R2: the values you found on paper
answer_25_R2a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_25_R2b = ...  # b) a number, 1 decimal
answer_25_R2c = ...  # c) True or False

for letter, answer in zip("abc", [answer_25_R2a, answer_25_R2b, answer_25_R2c]):
    wb.check(f"25.R2{letter}", answer)

**25.R3 — La divergence KL, ce que mesure le terme de régularisation du VAE**

In [ ]:
# 25.R3: the values you found on paper
answer_25_R3a = ...  # a) a number, 2 decimals
answer_25_R3b = ...  # b) True or False
answer_25_R3c = ...  # c) True or False
answer_25_R3d = ...  # d) True or False
answer_25_R3e = ...  # e) a number, 2 decimals

for letter, answer in zip("abcde", [answer_25_R3a, answer_25_R3b, answer_25_R3c, answer_25_R3d, answer_25_R3e]):
    wb.check(f"25.R3{letter}", answer)

**Ex 25.1 — Compter les paramètres d'un autoencodeur à goulot**

In [ ]:
# Ex 25.1: the values you found on paper
answer_25_1a = ...  # a) a whole number
answer_25_1b = ...  # b) a whole number
answer_25_1c = ...  # c) a whole number
answer_25_1d = ...  # d) a whole number
answer_25_1e = ...  # e) a whole number
answer_25_1f = ...  # f) a number, 1 decimal
answer_25_1g = ...  # g) a whole number

for letter, answer in zip("abcdefg", [answer_25_1a, answer_25_1b, answer_25_1c, answer_25_1d, answer_25_1e, answer_25_1f, answer_25_1g]):
    wb.check(f"25.1{letter}", answer)

**Ex 25.2 — Mélanger deux cercles : pixels contre paramètres**

In [ ]:
# Ex 25.2: the values you found on paper
answer_25_2a = ...  # a) a vector as a string, e.g. "(1, -2)"
answer_25_2b = ...  # b) a number, 2 decimals
answer_25_2c = ...  # c) a number, 2 decimals
answer_25_2d = ...  # d) a number, 2 decimals
answer_25_2e = ...  # e) a number, 2 decimals
answer_25_2f = ...  # f) a whole number
answer_25_2g = ...  # g) a number, 1 decimal
answer_25_2h = ...  # h) a number, 1 decimal

for letter, answer in zip("abcdefgh", [answer_25_2a, answer_25_2b, answer_25_2c, answer_25_2d, answer_25_2e, answer_25_2f, answer_25_2g, answer_25_2h]):
    wb.check(f"25.2{letter}", answer)

**Ex 25.3 — Suivre les formes dans l'autoencodeur convolutif**

In [ ]:
# Ex 25.3: the values you found on paper
answer_25_3a = ...  # a) a shape as a string, e.g. "(3, 5, 1)"
answer_25_3b = ...  # b) a shape as a string, e.g. "(3, 5, 1)"
answer_25_3c = ...  # c) a shape as a string, e.g. "(3, 5, 1)"
answer_25_3d = ...  # d) a whole number
answer_25_3e = ...  # e) a whole number
answer_25_3f = ...  # f) a whole number
answer_25_3g = ...  # g) a number, 2 decimals
answer_25_3h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_25_3a, answer_25_3b, answer_25_3c, answer_25_3d, answer_25_3e, answer_25_3f, answer_25_3g, answer_25_3h]):
    wb.check(f"25.3{letter}", answer)

**Ex 25.4 — Retrouver 28×28 : padding et output_padding d'une convolution transposée**

In [ ]:
# Ex 25.4: the values you found on paper
answer_25_4a = ...  # a) a whole number
answer_25_4b = ...  # b) a whole number
answer_25_4c = ...  # c) a whole number
answer_25_4d = ...  # d) a whole number
answer_25_4e = ...  # e) a whole number
answer_25_4f = ...  # f) a whole number
answer_25_4g = ...  # g) a whole number
answer_25_4h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_25_4a, answer_25_4b, answer_25_4c, answer_25_4d, answer_25_4e, answer_25_4f, answer_25_4g, answer_25_4h]):
    wb.check(f"25.4{letter}", answer)

**Ex 25.6 — Où tombent les tirages d'une gaussienne en grande dimension**

In [ ]:
# Ex 25.6: the values you found on paper
answer_25_6a = ...  # a) a whole number
answer_25_6b = ...  # b) a number, 2 decimals
answer_25_6c = ...  # c) a number, 3 decimals
answer_25_6d = ...  # d) a number, 3 decimals
answer_25_6e = ...  # e) a number, 2 decimals
answer_25_6f = ...  # f) a whole number
answer_25_6g = ...  # g) a number, 2 decimals
answer_25_6h = ...  # h) a number, 2 decimals

for letter, answer in zip("abcdefgh", [answer_25_6a, answer_25_6b, answer_25_6c, answer_25_6d, answer_25_6e, answer_25_6f, answer_25_6g, answer_25_6h]):
    wb.check(f"25.6{letter}", answer)

**Ex 25.7 — La divergence KL entre N(μ, σ²) et N(0, 1)**

In [ ]:
# Ex 25.7: the values you found on paper
answer_25_7a = ...  # a) a number, 3 decimals
answer_25_7b = ...  # b) a number, 3 decimals
answer_25_7c = ...  # c) a number, 3 decimals
answer_25_7d = ...  # d) a number, 3 decimals
answer_25_7e = ...  # e) a number, 3 decimals
answer_25_7f = ...  # f) a number, 3 decimals
answer_25_7g = ...  # g) a number, 3 decimals
answer_25_7h = ...  # h) a number, 3 decimals

for letter, answer in zip("abcdefgh", [answer_25_7a, answer_25_7b, answer_25_7c, answer_25_7d, answer_25_7e, answer_25_7f, answer_25_7g, answer_25_7h]):
    wb.check(f"25.7{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu expliquer le rôle du goulot, et pourquoi un autoencodeur entraîné sur une seule image ne comprime rien ?
2. Sais-tu dire ce que l'autoencodeur linéaire a en commun avec l'ACP, et ce qu'il ne retrouve pas ?
3. Sais-tu écrire la reparamétrisation et la KL gaussienne, et expliquer pourquoi un VAE sait générer ?

**Pour aller plus loin** : l'introduction aux VAE de Kingma et Welling (2019), citée dans la fiche. La suite : le ch. 26, l'apprentissage par renforcement, puis les GAN au ch. 27.

*Les exercices de code 25.14 à 25.39 (parties A à C : les autoencodeurs denses, convolutifs et débruiteurs, puis le VAE, ta librairie `vae.py` et le défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 25` pour obtenir le notebook complet.*